# Serving& Cost Optimization

**Module 14 · Lesson 14.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Measure First
- Prompt Caching
- KV Cache vs Prompt Cache
- Model Routing (Gated)
- Quantization (the Cliff)
- Batch API

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai -q

# Reproducibility - the lesson uses seeded random values throughout

## The bill lands, and it is five times the estimate

> 💡 **Info**
>
> Your LLM feature shipped, usage grew, and the invoice is **five times** what you modelled. The reflex — the wrong first move — is to open the biggest prompt you can find and start deleting words. It feels like progress and it saves almost nothing, because you have no idea where the money actually goes. This lesson is the opposite: a **measure-first playbook**. First, read the per-feature spend from the traces you already emit (14.1) and find the slice that is most of the bill — it is almost never the one you would have guessed. Then apply the right lever to *that* slice: **cache** the stable prefix, **route** the easy queries to a cheaper model, **quantize** the open model to a smaller GPU, **batch** the offline jobs. After each one, measure the delta — did the bill actually move? Two of those levers change what the user sees, so they pass through the **eval gate** from 14.3 before they ship. And when you have stacked them, one last question remains: at your volume, is it cheaper to keep renting the API or to buy the GPU — and does that answer survive the next price change? Master this loop and you turn a runaway bill into a line you control.

### 🎯 What you will be able to do after this lesson

- **Apply** a measure-first cost audit — aggregate per-feature spend from the traces and find the expensive slice *before* you touch a single prompt.

- **Build** each cost lever as a runnable model — prompt caching, KV-versus-prompt cache, gated routing, the quantization cliff, the batch discount, and the self-host crossover.

- **Compare** KV cache (within-request latency, not billed) with prompt cache (across-request cost, billed at a discount); realtime with batch; and renting the API with buying the GPU.

- **Evaluate** whether a cheaper route or a smaller quant is safe to ship (gate it on the golden set), and when the self-host decision flips as prices move.

> 📦 **Info**
>
> ✅ Before you startThis lesson stands on three earlier ones. In **14.1** every call emitted a trace with token counts and a cost; here you aggregate those into a per-feature bill. In **14.3** you built a golden set and a CI eval gate; here that same gate decides whether a cheaper model or a smaller quant is allowed to ship. And all the underlying mechanics — how caching works, what quantization does to a model, how vLLM batches, how the crossover is derived — are Module 13; this lesson *operates* them rather than re-deriving them. If a mechanism feels new, the reference is in 13.1 through 13.6.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🏡 **Analogy**
>
> Cutting an LLM bill is like **cutting a house’s energy bill**. The amateur move is to start unplugging phone chargers — it feels virtuous and changes nothing, because the bill is dominated by heating and the water heater. The professional runs an **energy audit** first: a thermal camera shows exactly where the heat leaks, and only then do you insulate *that* wall. Measuring the per-feature spend from your traces is the thermal camera; the levers in this lesson are the insulation. **Where it breaks down:** a house’s heat loss is roughly fixed, but an LLM’s cost landscape shifts under you — providers change prices, so you re-run the audit, you do not do it once.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Shorten the prompt to cut the bill.”** You optimised the cheap slice. The expensive retrieval-and-answer calls are the large majority of the bill from a small fraction of the calls; a shorter prompt on the many cheap calls barely registers. Measure first, then cut the slice that matters.
> - **“KV cache and prompt cache are the same thing.”** They are different caches on different axes. One reuses attention state *within* a request (a latency win, on the GPU, never billed); the other reuses a prefix *across* requests (a cost win, billed at a discount). Confusing them is the single most common interview miss in this area.

> 💡 **Info**
>
> ⚠️ Anti-patternThe cost win that quietly breaks the product: swapping in the cheapest model or the smallest quant to save money with *no eval gate*. The dashboard turns green — latency down, cost down, error rate flat — and nobody notices that the golden-set pass rate fell below the bar, because nobody ran it. You traded a real, measurable cost saving for a silent, unmeasured quality regression. Every quality-affecting lever in this lesson (routing, quantization) passes through the same gate a prompt change does.

---

## 🎯 Concept 1: Measure First

### Measure First

You cannot cut what you cannot see. Before optimising anything, aggregate the per-feature cost from your traces (14.1) and find the Pareto: a small slice of calls almost always drives most of the bill. Optimise THAT slice - not the many cheap calls that are easy to see but cost nothing.

Every cost lever in this lesson is useless if you point it at the wrong slice, so the playbook starts with measurement. You already emit the raw material: in Lesson 14.1 every call wrote a trace carrying token counts and a computed cost. Aggregate those by feature and a **Pareto** appears — a small number of expensive calls dominate the total, while a large number of cheap calls barely register. In the worked example, a retrieval-and-answer feature is the *large majority* of the bill from only a *quarter* of the calls, while a classify feature is the majority of the *calls* but a rounding error on *cost*. The lesson is brutal and simple: shaving tokens off the many cheap classify calls saves almost nothing, and halving the cost of the one expensive feature saves most of the bill. This is why every later step in this lesson is preceded, in practice, by this one. The block builds the per-feature breakdown from a trace log and finds the slice worth attacking, keyless.

> 📈 **Analogy**
>
> Measuring first is **the energy audit before you buy insulation**. You would not re-insulate a random wall because it was easy to reach — you point a thermal camera at the whole house and it shows, unmistakably, that one wall and the roof are where the heat pours out. Only then do you spend. The per-feature spend breakdown is that thermal image: it tells you, before you spend a single hour optimising, that one feature is the whole problem and the busy little classify calls are a distraction. Optimise where the heat is leaking, not where it is convenient to look.

Your trace log shows one expensive feature is the large majority of the bill from a quarter of the calls, and a cheap feature is most of the calls but a tiny slice of cost. Where do you optimise?

**📝 `01_measure_first.py`** - *runnable*

In [ ]:
# You cannot cut what you cannot see. Before optimising, aggregate the per-feature cost from your traces (Lesson 14.1)
# and find the Pareto: a small slice of calls usually drives most of the bill. Optimise THAT, not the cheap stuff.
traces = [("classify", 120, 0.0003), ("rag_answer", 50, 0.030), ("summarize", 30, 0.004)]  # (feature, calls, $/call) illustrative
total = sum(calls * cost for _, calls, cost in traces)
total_calls = sum(calls for _, calls, cost in traces)
print("Per-feature cost (from the trace log):")
for feat, calls, cost in sorted(traces, key=lambda t: -t[1] * t[2]):
    spend = calls * cost
    print("  {:<12} {:>4} calls  ${:.4f}/call  -> ${:.4f}  ({:.0%} of bill, {:.0%} of calls)".format(
          feat, calls, cost, spend, spend / total, calls / total_calls))
top = max(traces, key=lambda t: t[1] * t[2])
print()
print("The {} feature is {:.0%} of the bill from just {:.0%} of the calls - that is where the money is.".format(
      top[0], top[1] * top[2] / total, top[1] / total_calls))
print("Optimise the expensive slice; shaving the cheap classify calls would save almost nothing. Measure first.")

# Output:
# Per-feature cost (from the trace log):
#   rag_answer     50 calls  $0.0300/call  -> $1.5000  (91% of bill, 25% of calls)
#   summarize      30 calls  $0.0040/call  -> $0.1200  (7% of bill, 15% of calls)
#   classify      120 calls  $0.0003/call  -> $0.0360  (2% of bill, 60% of calls)
#
# The rag_answer feature is 91% of the bill from just 25% of the calls - that is where the money is.
# Optimise the expensive slice; shaving the cheap classify calls would save almost nothing. Measure first.

- Each feature’s spend is **calls times cost-per-call**, aggregated from the trace log (14.1).
- One feature is the **large majority of the bill** from only a quarter of the calls — the Pareto.
- The busiest feature by call count is a rounding error on cost; trimming it would save almost nothing.
- You cannot cut what you cannot see — measure per-feature first, then optimise the expensive slice.

#### 💡 What just happened

⚡ What just happened?Before touching anything, aggregate per-feature cost from the traces and find the Pareto: one slice is most of the bill from a fraction of the calls. Optimise that slice, not the cheap-but-numerous calls. Tradeoff: a few minutes reading traces, in exchange for spending your effort where the money actually is. Next: the first and cheapest lever - caching the stable prefix.

- A per-feature spend Pareto where one slice dwarfs the rest
- Play reveals the bars and the large-majority-of-bill-from-a-quarter-of-calls split

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Prompt Caching

### Prompt Caching

A request is a stable PREFIX (system + few-shot + retrieved docs) plus a variable SUFFIX (the user query). Provider caching bills the prefix at a deep discount after the first call - Anthropic reads at ten percent of input, OpenAI auto-caches at half. The rule that makes it work: keep the variable part LAST, so the prefix stays identical and cacheable.

The cheapest lever touches no model choice at all: stop paying full price for tokens you send on every single call. A production request is almost always a large **stable prefix** — system instructions, few-shot examples, retrieved policy documents — followed by a small **variable suffix**, the user’s actual query. **Prompt caching** lets the provider store that prefix after the first call and bill it at a deep discount on every subsequent one: on Anthropic a cache read is about *ten percent* of the normal input price (the first call pays a small write premium), and on OpenAI automatic caching takes *half* off the cached prefix. In the worked example a warm call costs a small fraction of a cold one — roughly an *eighty-seven percent* saving per call — and over a hundred calls the run is cheaper by a similar margin. The one rule you must not break: keep the **variable part last**. If anything changes early in the prompt, the prefix is no longer identical, the cache misses, and you are back to full price. One ops condition the warm-case math hides: provider prompt caches live on a short **expiry window** — Anthropic’s `ephemeral` cache defaults to about five minutes, refreshed on each hit (a one-hour option exists) — so the read discount only lands while calls keep arriving fast enough to keep the prefix warm. A steady stream of traffic reaps it; a cold or bursty workload with long gaps re-writes the cache and pays the write premium without ever collecting the reads. The block models a cached prefix against a variable suffix and totals a hundred calls under a warm stream, keyless; the illustrative artifact below is the real Anthropic call that marks the prefix cacheable.

> 🥦 **Analogy**
>
> Prompt caching is a chef’s **mise en place: chop once, reuse all night**. A good kitchen does not dice onions from scratch for every order — before service, the cook preps the stable ingredients once and keeps them at the station, so each dish only adds the parts that actually differ. The stable prefix is the mise en place: you pay to prep it once (the cache write), then every order reuses it at almost no cost (the cache read), and you only do fresh work for the one thing that changes per order — the user’s query. And just as a cook keeps the prepped items untouched, you keep the prefix identical: re-chop it mid-service and the whole advantage is gone.

You cache a request with a big stable prefix and a small user query. To keep the cache working, where does the variable user query go?

**📝 `02_prompt_caching.py`** - *runnable*

In [ ]:
# PROMPT CACHING: a request is a stable PREFIX (system + few-shot + retrieved docs) plus a variable SUFFIX (the user
# query). Provider caching bills the prefix at a deep discount after the first call - so put the variable part LAST.
PRICE_IN = 3.00          # $/1M normal input tokens (illustrative)
CACHE_READ = 0.30        # Anthropic: cache reads = 10% of input (a 90% discount)
CACHE_WRITE = 3.75       # Anthropic: cache writes = 1.25x input (paid once, on the first call)
prefix, suffix = 1800, 60   # tokens: the stable prefix vs the variable user query
no_cache = (prefix + suffix) * PRICE_IN / 1e6
first_call = (prefix * CACHE_WRITE + suffix * PRICE_IN) / 1e6   # writes the cache
warm_call = (prefix * CACHE_READ + suffix * PRICE_IN) / 1e6     # reads the cache
K = 100
no_cache_total = K * no_cache
cache_total = first_call + (K - 1) * warm_call
print("Per call: no-cache ${:.6f}   first(write) ${:.6f}   warm(read) ${:.6f}".format(no_cache, first_call, warm_call))
print("A warm call is {:.0%} of a no-cache call - about {:.0%} off, because the prefix drops to 10% of input.".format(warm_call / no_cache, 1 - warm_call / no_cache))
print("Over {} calls: no-cache ${:.4f}  vs  cached ${:.4f}  ->  {:.0%} cheaper.".format(K, no_cache_total, cache_total, 1 - cache_total / no_cache_total))
print()
print("Cache the stable prefix, pay full price only for the suffix. Change the prefix and you bust the cache - keep the")
print("variable part LAST. (Anthropic reads = 10% of input / writes 1.25x; OpenAI auto-caching = 50% off the cached prefix.)")

# Output:
# Per call: no-cache $0.005580   first(write) $0.006930   warm(read) $0.000720
# A warm call is 13% of a no-cache call - about 87% off, because the prefix drops to 10% of input.
# Over 100 calls: no-cache $0.5580  vs  cached $0.0782  ->  86% cheaper.
#
# Cache the stable prefix, pay full price only for the suffix. Change the prefix and you bust the cache - keep the
# variable part LAST. (Anthropic reads = 10% of input / writes 1.25x; OpenAI auto-caching = 50% off the cached prefix.)

**📝 `prompt_cache_real.py`** - *illustrative*

In [ ]:
# PRODUCTION PROMPT CACHING - an illustrative minimal example (needs the Anthropic SDK + a key; not run here).
import os
from anthropic import Anthropic
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"], timeout=30)

# The STABLE prefix: system instructions + few-shot + retrieved policy docs (~1800 tokens, identical every call).
SYSTEM_PREFIX = "You are a support agent. Policy: ...<about 1800 tokens of stable instructions + examples>..."

def answer(user_query):
    resp = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=400,
        system=[{"type": "text", "text": SYSTEM_PREFIX,
                 "cache_control": {"type": "ephemeral"}}],    # mark the prefix cacheable
        messages=[{"role": "user", "content": user_query}])   # the VARIABLE part stays LAST, never cached
    u = resp.usage
    # First call WRITES the cache (u.cache_creation_input_tokens, billed at 1.25x once); every warm call
    # READS it (u.cache_read_input_tokens, billed at 10% of input). Change the prefix and you bust the cache.
    return resp.content[0].text, u.cache_creation_input_tokens, u.cache_read_input_tokens
# Output: (illustrative - needs the Anthropic SDK + ANTHROPIC_API_KEY; a real cached call, not run here.)

- A request splits into a large **stable prefix** and a small **variable suffix** (the user query).
- The first call **writes** the cache at a small premium; every warm call **reads** it at about a tenth of the input price.
- A warm call costs a small fraction of a cold one — a large per-call saving that compounds over a warm run.
- The **illustrative artifact** is the real call: it marks the prefix cacheable with `cache_control` and keeps the user query last; the write discount only pays off while traffic keeps the ~five-minute cache warm.
- Keep the variable part **last**: change the prefix and you bust the cache and pay full price again.

#### 💡 What just happened

⚡ What just happened?Prompt caching bills the stable prefix at a deep discount (Anthropic reads at a tenth of input, OpenAI at half) after the first call - keep the variable part last so the prefix stays cacheable. Tradeoff: a small write premium on the first call, repaid many times over. Next: a cache that is easy to confuse with this one but does something completely different.

- A request split into a big cached prefix and a tiny variable suffix
- A calls slider drops the cost; a bust-the-cache toggle reverts it to full price

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: KV Cache vs Prompt Cache

### KV Cache vs Prompt Cache

The single most common mixup in this area. KV cache reuses attention state WITHIN one request’s decode - a latency win, per-request, on the GPU, and NOT billed. Prompt cache reuses a prefix ACROSS requests - a cost win, billed at a discount. Different caches, different axes, and you want both. Confusing them is a classic interview miss.

Step 2 was a *cost* optimisation across requests. There is a second cache with a nearly identical name that is a *latency* optimisation within a single request, and mixing them up is the classic interview stumble. The **KV cache** lives inside the model’s decode loop: as it generates each output token, it would otherwise recompute attention over every previous token, which grows with the square of the length; caching the keys and values makes each step reuse the prior work, so the total collapses from quadratic to linear. In the worked example, generating a couple hundred tokens naively costs on the order of twenty thousand attention-steps, while the KV-cached version costs a couple hundred — roughly a hundred times less decode work. But note what it is and is not: its scope is **one request**, its effect is **faster decode** (latency), it lives **on the GPU**, and it is **not billed** as a line item — you met its mechanics in Lesson 13.4. The **prompt cache** from Step 2 is the opposite on every axis: scope **across requests**, effect **cheaper input** (cost), and **billed** at a discount. You want both, and you should never confuse them. The block contrasts the two on their own terms, keyless.

> 👷 **Analogy**
>
> The two caches are **a chef reusing prep within one dish versus the shared pantry across every dish**. While plating a single complicated dish, the cook keeps the components they have already made within arm’s reach so they are not remade at each step — that is the KV cache, speeding up *this one dish*, invisible on the bill. The shared pantry of pre-prepped staples that *every* order draws from at a discount is the prompt cache — it lowers the *cost* across many dishes. One makes a single plate come out faster; the other makes every plate cheaper. Same word ‘cache’, two completely different kitchens.

Which statement correctly separates the two caches?

**📝 `03_kv_vs_prompt_cache.py`** - *runnable*

In [ ]:
# KV CACHE vs PROMPT CACHE - the #1 mixup. They are DIFFERENT caches on DIFFERENT axes. KV cache reuses attention state
# WITHIN one request's decode (a latency win, per-request, on the GPU, not billed). Prompt cache reuses a prefix ACROSS
# requests (a cost win, billed at a discount). You want BOTH; they do not overlap.
n = 200                                   # output tokens generated in one request
naive_attn = n * (n + 1) // 2             # recompute attention over ALL prior tokens at every step (O(n^2))
kv_attn = n                               # KV cache: reuse the cached keys/values, one step of work per token (O(n))
print("WITHIN one request (KV cache), generating {} tokens:".format(n))
print("  naive recompute: {:>6} attention-steps   KV cache: {:>4} steps   -> about {:.0f}x less decode work".format(naive_attn, kv_attn, naive_attn / kv_attn))
print("  scope: one request | effect: faster decode (latency) | lives: on the GPU | billed: no")
print()
print("ACROSS requests (prompt cache), a shared 1800-token prefix reused by many users:")
print("  effect: the input is billed at a discount (10% of input on Anthropic) | scope: many requests | billed: yes, discounted")
print()
print("So: KV cache saves TIME within a request; prompt cache saves MONEY across requests. Different caches, both worth having.")

# Output:
# WITHIN one request (KV cache), generating 200 tokens:
#   naive recompute:  20100 attention-steps   KV cache:  200 steps   -> about 100x less decode work
#   scope: one request | effect: faster decode (latency) | lives: on the GPU | billed: no
#
# ACROSS requests (prompt cache), a shared 1800-token prefix reused by many users:
#   effect: the input is billed at a discount (10% of input on Anthropic) | scope: many requests | billed: yes, discounted
#
# So: KV cache saves TIME within a request; prompt cache saves MONEY across requests. Different caches, both worth having.

- **KV cache**: within one request, decode work drops from quadratic to linear — here about a hundred times less.
- Its axes: scope one request, effect faster decode (latency), lives on the GPU, **not billed**.
- **Prompt cache**: across many requests, the shared prefix is billed at a discount — a cost win, **billed**.
- Different caches on different axes; you want both, and confusing them is the classic miss.

#### 💡 What just happened

⚡ What just happened?KV cache reuses attention state within one request (latency, on the GPU, not billed); prompt cache reuses a prefix across requests (cost, billed at a discount). Same word, opposite axes - hold them apart. Tradeoff: none - they are complementary. Next: the first lever that changes what the user sees, so it needs a gate.

- Two panels: a within-request decode-work curve beside an across-request cost discount
- An output-tokens slider grows the quadratic-vs-linear gap

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Model Routing (Gated)

### Model Routing (Gated)

Send easy queries to a cheap model and hard ones to a strong model, and the blended cost drops sharply. But routing is a QUALITY decision, not just a cost one: the cheap route must clear the eval gate (14.3) - its golden-set pass rate above the bar - or you have traded a real saving for silent quality loss. Route on difficulty, measure PER-ROUTE, and gate it.

The levers so far were free: caching costs no quality. Routing is the first lever that *can* cost quality, so it changes how you ship it. The idea is simple and powerful: most traffic is easy, and a cheap model handles easy queries fine, so send the easy ones to the cheap model and reserve the strong, expensive model for the hard ones. In the worked example, splitting the traffic cuts the blended cost by about *half* versus sending everything to the strong model — a large saving for a change nobody sees on the easy queries. The trap is treating that as a pure cost decision. A cheaper model on a query it *cannot* actually handle does not error — it returns a confident, worse answer, and your dashboard stays green. So routing goes through the **eval gate** from Lesson 14.3: you measure the cheap route’s pass rate on the golden set, and you only ship the route if it stays **above the quality bar**. In the example, a cheap route that scores above the bar is safe; one that scores below it is *blocked*, because the saving is not worth the silent regression. The cheapest usable models here span Google, OpenAI, and Anthropic, and the routing mechanics themselves are Lesson 13.3 — here the new part is the gate. The block routes a thousand queries and gates the cheap lane, keyless.

> 🏥 **Analogy**
>
> Routing is **a triage nurse at the front of the clinic**. The nurse does not send every patient to the senior consultant — that would be ruinously slow and expensive. They quickly sort: the routine cases go to the general clinic (cheap, fast, perfectly adequate), and only the genuinely complex ones go to the specialist. That is the cost win. But a good triage system is *audited*: someone checks that the routine lane is not quietly missing serious cases. That audit is the eval gate — the cheap route is only allowed to keep taking patients as long as its outcomes stay above the quality bar. Triage without the audit is how a cheap route silently harms the cases it should have escalated.

You route easy queries to a cheap model and the blended cost drops by about half. What has to be true before you ship the route?

**📝 `04_model_routing.py`** - *runnable*

In [ ]:
# MODEL ROUTING: send easy queries to a cheap model and hard ones to a strong model. It cuts the blended cost - but it
# is a QUALITY decision: the cheap route must clear the eval gate (Lesson 14.3) or you have traded cost for silent quality loss.
easy, hard = 700, 300                     # of 1000 queries
OUT = 200                                 # output tokens per query (illustrative; output dominates cost)
CHEAP_OUT, STRONG_OUT = 4.00, 15.00       # $/1M output for a cheap vs a strong model
all_strong = (easy + hard) * OUT * STRONG_OUT / 1e6
routed = (easy * OUT * CHEAP_OUT + hard * OUT * STRONG_OUT) / 1e6
print("All-strong: ${:.4f}    Routed (easy->cheap, hard->strong): ${:.4f}  ->  {:.0%} cheaper.".format(all_strong, routed, 1 - routed / all_strong))
BAR = 0.92
cheap_route_passrate = 0.93               # measured on the golden set for the easy slice
print()
print("But routing is only valid if the cheap route CLEARS THE GATE. Cheap-route pass-rate on the golden set: {:.0%}.".format(cheap_route_passrate))
print("  {:.0%} >= the {:.0%} bar -> route is safe.  (A cheap route at 0.88 < 0.92 would trade the {:.0%} saving for silent quality loss.)".format(
      cheap_route_passrate, BAR, 1 - routed / all_strong))
print("Route on a difficulty signal, measure PER-ROUTE quality, and gate it. (Routing mechanics: Lesson 13.3.)")

# Output:
# All-strong: $3.0000    Routed (easy->cheap, hard->strong): $1.4600  ->  51% cheaper.
#
# But routing is only valid if the cheap route CLEARS THE GATE. Cheap-route pass-rate on the golden set: 93%.
#   93% >= the 92% bar -> route is safe.  (A cheap route at 0.88 < 0.92 would trade the 51% saving for silent quality loss.)
# Route on a difficulty signal, measure PER-ROUTE quality, and gate it. (Routing mechanics: Lesson 13.3.)

- Easy queries go to the **cheap** lane, hard queries to the **strong** lane; the blended cost is roughly half of all-strong.
- But routing is a **quality decision**: the cheap route’s pass rate is measured on the golden set.
- Above the quality bar the route is **safe**; below it the route is **blocked**, because the saving is not worth silent quality loss.
- Route on difficulty, measure per-route, and gate it (routing mechanics: 13.3; the gate: 14.3).

#### 💡 What just happened

⚡ What just happened?Routing sends easy queries to a cheap model and hard ones to a strong model, cutting blended cost by about half - but it is a quality decision, so the cheap route must clear the eval gate or it is blocked. Tradeoff: a routing signal and a golden-set check, in exchange for a large saving that does not degrade quality. Next: the same gate applied to shrinking an open model.

- Queries flow into a cheap and a strong lane with a blended-cost bar
- A gate checks the cheap lane against the quality bar - safe or blocked

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Quantization (the Cliff)

### Quantization (the Cliff)

Quantization shrinks an open model - VRAM equals params times bits over eight, so four-bit is about four times smaller than sixteen-bit - which lets it run on a cheaper GPU. But quality does not degrade smoothly: it falls off a CLIFF at low bit-widths. Measure quality on the golden set (14.3) and pick the SMALLEST quant that still clears the bar. About four-bit is the usual safe floor.

If self-hosting an open model, quantization is the lever that decides which GPU you can afford. Storing each weight in fewer bits shrinks the model: the VRAM it needs is roughly its parameter count times the bits per weight over eight, so moving from sixteen-bit to four-bit makes a model about *four times smaller* and lets it run on a far cheaper card. The catch is that quality does not fall gracefully as you drop bits — it holds nearly flat for a while and then falls off a **cliff**. In the worked example a seven-billion-parameter model keeps almost all of its quality down to four-bit, then drops sharply at three-bit and collapses at two-bit. So the decision is not “go as small as possible” — that walks straight off the cliff — it is “pick the **smallest quant that still clears the golden-set bar**.” That makes quantization a *quality* decision, gated on the exact same golden set from Lesson 14.3 that gated routing: you measure each quant’s pass rate and choose the smallest one above the bar. About four-bit is the usual safe floor. The mechanics of quantization formats live in Lesson 13.5; here the new part, again, is the gate. The block sweeps the quant levels, computes each one’s VRAM, and picks the smallest above the bar, keyless.

> 📷 **Analogy**
>
> Quantization is **JPEG compression until the image visibly breaks**. Saving a photo at a lower quality setting makes the file dramatically smaller, and for a long way down the image looks essentially identical — you are getting the space saving for free. But there is a point where blocky artifacts suddenly appear and the photo is ruined; the quality does not fade evenly, it cliffs. Quantizing a model is the same: you keep shrinking the file and the answers stay good, right up until a bit-width where they abruptly fall apart. The craft is finding the smallest file that still looks right — and for a model, ‘looks right’ is a number: the golden-set pass rate above the bar.

You quantize an open model and quality holds flat down to four-bit, then drops sharply at three-bit. Which quant do you ship?

**📝 `05_quantization.py`** - *runnable*

In [ ]:
# QUANTIZATION shrinks an open model (VRAM = params x bits / 8) - roughly 4x smaller at 4-bit - but quality has a CLIFF.
# Do not eyeball it: measure quality on the golden set (Lesson 14.3) and pick the SMALLEST quant that still clears the bar.
PARAMS_B = 7                              # a 7B open model
QUANTS = [("FP16", 16, 1.00), ("Q8_0", 8, 0.99), ("Q4_K_M", 4, 0.96), ("Q3_K", 3, 0.88), ("Q2_K", 2, 0.71)]  # illustrative quality
BAR = 0.92
print("quant     bits   VRAM      golden-set quality   clears {:.0%} bar?".format(BAR))
chosen = None
for name, bits, quality in QUANTS:
    vram = PARAMS_B * bits / 8
    ok = quality >= BAR
    if ok:
        chosen = (name, vram)
    print("  {:<7}  {:>2}    {:>4.1f} GB   {:.0%}                 {}".format(name, bits, vram, quality, "yes" if ok else "NO (below the cliff)"))
fp16_vram = PARAMS_B * 16 / 8
print()
print("The smallest quant that still clears the bar is {} at {:.1f} GB - about {:.0f}x less VRAM than FP16.".format(chosen[0], chosen[1], fp16_vram / chosen[1]))
print("Below it (Q3, Q2) quality falls off a cliff: you save VRAM but lose the product. Q4_K_M ~96% is the usual safe floor.")
print("Quantization is a QUALITY decision too - gate it on the golden set, exactly like routing. (Mechanics: Lesson 13.5.)")

# Output:
# quant     bits   VRAM      golden-set quality   clears 92% bar?
#   FP16     16    14.0 GB   100%                 yes
#   Q8_0      8     7.0 GB   99%                 yes
#   Q4_K_M    4     3.5 GB   96%                 yes
#   Q3_K      3     2.6 GB   88%                 NO (below the cliff)
#   Q2_K      2     1.8 GB   71%                 NO (below the cliff)
#
# The smallest quant that still clears the bar is Q4_K_M at 3.5 GB - about 4x less VRAM than FP16.
# Below it (Q3, Q2) quality falls off a cliff: you save VRAM but lose the product. Q4_K_M ~96% is the usual safe floor.
# Quantization is a QUALITY decision too - gate it on the golden set, exactly like routing. (Mechanics: Lesson 13.5.)

- VRAM is **params times bits over eight**, so four-bit is about four times smaller than sixteen-bit.
- Quality holds nearly flat down to four-bit, then **falls off a cliff** at three- and two-bit.
- The choice is the **smallest quant that still clears the golden-set bar** — here about four-bit.
- Quantization is a quality decision, gated on the golden set exactly like routing (mechanics: 13.5).

#### 💡 What just happened

⚡ What just happened?Quantization shrinks an open model (VRAM = params times bits over eight) but quality falls off a cliff at low bit-widths; pick the smallest quant that still clears the golden-set bar, about four-bit as the usual floor. Tradeoff: a little quality for a lot less VRAM, but only down to the bar. Next: a lever for work nobody is waiting on.

- A quality-versus-bits curve with a golden-set bar line
- The low-bit points fall off a cliff below the bar

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Batch API

### Batch API

Work that can WAIT - it returns within a window, up to twenty-four hours - is billed at about half price on both Anthropic and OpenAI. Use it for offline jobs: backfills, bulk summaries, even running your eval suite. It is not for anything a user is watching. And it STACKS: batch (half off) on top of a cache read (about ninety percent off) reaches roughly ninety-five percent off the repeated portion.

Not all work is urgent, and the provider will pay you to wait. The **batch API** takes a large job, runs it asynchronously, and returns the results within a window — up to twenty-four hours — for about *half* the realtime price, on both Anthropic and OpenAI. The only question you have to answer is: **is a user waiting on this?** If yes, it is realtime and you pay full price for the immediacy. If no — backfilling summaries for a search index, regenerating embeddings, classifying a data dump, or running your own nightly eval suite from Lesson 14.3 — it goes to batch and costs half. In the worked example, backfilling fifty thousand offline summaries costs full price at realtime and exactly half that on batch. The real power is that this lever **stacks** with the others: a batch job (half off) whose calls also hit a warm prompt cache (about ninety percent off the prefix) lands at roughly *ninety-five percent* off the repeated portion. Stacking cheap levers, not finding one magic lever, is how a five-times overrun becomes affordable. The block prices a realtime backfill against a batch one, keyless.

> 🧼 **Analogy**
>
> The batch API is **wash-and-fold laundry: half price if you can wait**. Same-day service costs a premium because someone drops everything to do your load now; if you are happy to pick it up tomorrow, the shop schedules it into the quiet hours and charges you much less — identical clean clothes, just not immediately. The batch API is exactly that deal for tokens: the offline backfill you do not need until tomorrow gets scheduled into the provider’s spare capacity at half price. The only mistake is sending same-day laundry — a user staring at a spinner — through the wash-and-fold queue.

You need to backfill fifty thousand summaries for a search index that ships next week. Realtime or batch?

**📝 `06_batch_api.py`** - *runnable*

In [ ]:
# BATCH API: work that can WAIT (returns within a window, up to 24h) is billed at ~50% off on both Anthropic and OpenAI.
# Use it for offline jobs - backfills, summaries, even running your eval suite - and stack it with cache reads.
n_jobs = 50_000                          # offline summaries to backfill
IN_TOK, OUT_TOK = 500, 150
PRICE_IN, PRICE_OUT = 3.00, 15.00        # $/1M
realtime_each = (IN_TOK * PRICE_IN + OUT_TOK * PRICE_OUT) / 1e6
realtime_total = n_jobs * realtime_each
batch_total = realtime_total * 0.50      # batch = 50% off
print("Backfilling {:,} summaries:".format(n_jobs))
print("  realtime: ${:.2f}    batch (50% off, async): ${:.2f}    -> saves ${:.2f}".format(realtime_total, batch_total, realtime_total - batch_total))
print()
print("Batch is for work a USER is not waiting on. Realtime for anything user-facing; batch for backfills, offline")
print("summaries, and running evals. Stack batch (50% off) with a cache read (90% off) -> about 95% off the repeated portion.")

# Output:
# Backfilling 50,000 summaries:
#   realtime: $187.50    batch (50% off, async): $93.75    -> saves $93.75
#
# Batch is for work a USER is not waiting on. Realtime for anything user-facing; batch for backfills, offline
# summaries, and running evals. Stack batch (50% off) with a cache read (90% off) -> about 95% off the repeated portion.

- A large offline job (fifty thousand summaries) is priced at realtime and at batch.
- Batch is about **half price** for work that returns within the window (up to twenty-four hours).
- The rule is simply *is a user waiting?* — realtime for user-facing, batch for backfills, bulk jobs, and evals.
- It **stacks**: batch on top of a warm cache read reaches roughly ninety-five percent off the repeated portion.

#### 💡 What just happened

⚡ What just happened?The batch API bills async work - anything a user is not waiting on - at about half price within a twenty-four-hour window, and it stacks with a cache read to reach roughly ninety-five percent off the repeated portion. Tradeoff: latency you do not need to spend, for half the bill. Next: the biggest decision of all - rent the API or buy the GPU.

- A realtime-versus-batch cost pair with a can-it-wait toggle
- The toggle routes a job to the right lane; a stack-with-cache readout

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Self-Host Crossover

### Self-Host Crossover

An API is a variable per-token cost; a GPU is a fixed monthly cost. The break-even volume is GPU cost divided by API price-per-token - below it the API is cheaper, above it the GPU wins - and utilization decides where you sit. Crucially, prices are NOT fixed: a vendor doubling its price HALVES the crossover, so self-hosting gets attractive at half the volume. Re-decide when prices move; do not hardcode one vendor.

The final decision ties the whole lesson together: at your volume, is it cheaper to keep *renting* the API or to *buy* the GPU? The shapes are different. An API bill is **variable** — it scales linearly with tokens, and at zero traffic it is zero. A rented GPU is a **fixed** monthly cost — you pay the same whether it runs flat out or sits idle. Draw both against monthly volume and they cross at a single point: the **break-even volume**, which is simply the GPU’s fixed cost divided by the API’s price per token. Below that volume the API is cheaper (you are not using enough of the GPU to justify it); above it the GPU wins. Which is why **utilization decides** — an idle GPU loses to every API, and a saturated one beats them. But the decision is not permanent, and this is the part people get wrong: **prices move**. Budget models keep getting cheaper, but in 2026 a frontier reasoning model roughly *doubled* its price — and a doubled API price *halves* the crossover, making self-hosting attractive at half the volume you needed before. So you re-run this number when prices change, in either direction, and you do not hardcode a single vendor forever. Platform-scale serving — multi-region, autoscaled inference behind an enterprise platform — is where this goes next, in Lesson 17.4. The block computes the crossover and then moves the price to show the line shift, keyless.

> 🚗 **Analogy**
>
> The crossover is **buy versus rent a car, and the break-even moves with the price of gas**. Renting is pure variable cost — you pay only for the days you drive, ideal if you rarely need a car. Buying is a fixed cost that only pays off past enough miles per month. There is a break-even mileage where they meet, and below it renting wins. But that line is not fixed: if the rental company doubles its daily rate, the mileage at which buying pays off drops sharply — suddenly owning makes sense for a lighter driver. The API is the rental, the GPU is the purchase, and a provider price change slides the break-even just like a change in the rental rate. You re-check the math when the price moves; you do not decide once and forget.

Your self-host crossover sits at some monthly volume. The API vendor doubles its price per token. What happens to the crossover?

**📝 `07_self_host_crossover.py`** - *runnable*

In [ ]:
# SELF-HOST CROSSOVER: an API is variable per-token; a GPU is a fixed monthly cost. Break-even volume = GPU cost / API
# price-per-token. Utilization decides (Lessons 13.5/13.6). And prices are NOT fixed - a rise MOVES the crossover, so re-decide.
gpu_monthly = 2000.0                      # a rented GPU box, $/month (illustrative)
api_price_per_1m = 3.00                   # $/1M input tokens today
def crossover_tokens(price_per_1m):
    return gpu_monthly / (price_per_1m / 1e6)   # tokens/month where GPU cost == API cost
base = crossover_tokens(api_price_per_1m)
print("At ${:.2f}/1M, self-hosting breaks even at ~{:.0f}M tokens/month; below that the API is cheaper, above it the GPU wins.".format(api_price_per_1m, base / 1e6))
doubled = 6.00                           # a frontier model doubled its price in 2026 - price RISES happen, not just cuts
new = crossover_tokens(doubled)
print("If the vendor DOUBLES the price to ${:.2f}/1M (this happened in 2026), the crossover HALVES to ~{:.0f}M tokens/month".format(doubled, new / 1e6))
print("  -> self-hosting becomes attractive at half the volume. The buy-vs-rent decision is NOT permanent.")
print()
print("Crossover = GPU fixed cost / API price-per-token; utilization decides. But re-run it when prices move (up or down),")
print("and do not hardcode one vendor forever - the ~100x spread from cheap models to the frontier keeps shifting.")

# Output:
# At $3.00/1M, self-hosting breaks even at ~667M tokens/month; below that the API is cheaper, above it the GPU wins.
# If the vendor DOUBLES the price to $6.00/1M (this happened in 2026), the crossover HALVES to ~333M tokens/month
#   -> self-hosting becomes attractive at half the volume. The buy-vs-rent decision is NOT permanent.
#
# Crossover = GPU fixed cost / API price-per-token; utilization decides. But re-run it when prices move (up or down),
# and do not hardcode one vendor forever - the ~100x spread from cheap models to the frontier keeps shifting.

- An API is a **variable** per-token cost; a GPU is a **fixed** monthly cost.
- They cross at the **break-even volume = GPU fixed cost / API price-per-token**; below it the API wins, above it the GPU.
- Utilization decides where you sit — an idle GPU loses to every API, a saturated one beats them.
- Doubling the API price **halves the crossover** — prices move, so re-decide and do not hardcode one vendor.

#### 💡 What just happened

⚡ What just happened?The self-host crossover is GPU fixed cost over API price-per-token; below it rent, above it buy, and utilization decides. Prices move - a doubled API price halves the crossover - so re-run the decision, do not carve it. Tradeoff: fixed cost and ops overhead against per-token freedom. That closes the loop: measure, apply, measure the delta, gate the risky levers, stack them, and re-decide when prices move.

- A sloped API line versus a flat GPU line, crossing at the break-even volume
- An API-price slider doubles the price and slides the crossover left

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — the cost-optimization ops loop
> A runaway bill is not cut by deleting words from a prompt; it is cut by a loop. **Measure** the per-feature spend from the traces and find the expensive slice (Step 1). **Apply** the right lever to that slice: cache the stable prefix (Step 2), and know the difference from the within-request KV cache (Step 3); route the easy queries to a cheaper model (Step 4); quantize an open model to a smaller GPU (Step 5); batch the offline work (Step 6). After each one, **measure the delta** — did the bill actually move? For the two levers that change what the user sees, routing and quantization, **gate** on the golden set from 14.3 before shipping. **Stack** the levers — batch on a cache read reaches most of the way to free on the repeated portion. And because prices move, **re-decide** the rent-versus-buy crossover whenever a vendor changes its price (Step 7). Ask two questions of any expensive LLM feature: do you know which slice of it is the money, and is every quality-affecting cut gated the way a prompt change is?

**📦 **The lever table****

| Lever | What it saves | Rough size | Needs an eval gate? |
|---|---|---|---|
| Measure first | points you at the expensive slice | prerequisite for all the rest | no |
| Prompt caching | the repeated stable prefix | up to ~10x on cache-heavy prompts | no |
| KV cache | decode latency, not the bill | within-request only, not billed | no |
| Model routing | blended cost across queries | about two to five times | YES(golden set) |
| Quantization | VRAM / the GPU you can rent | two to four times smaller | YES(golden set) |
| Batch API | async, non-urgent work | about half price, and it stacks | no |
| Self-host | per-token cost at high volume | depends on utilization | re-decide on price change |

The lesson’s real claim is that *stacking* cheap levers — not any single one — is what brings a five-times overrun back under control. Here it is, computed: start from the measured baseline (Step 1), cache the expensive slice’s prefix (Step 2), gate-route its easy half (Step 4), and batch the offline slice (Step 6), each lever on the slice it actually fits.

**📝 `08_the_stack.py`** - *runnable*

In [ ]:
# THE STACK: the levers are not either/or - you STACK them, each on the RIGHT slice from the measure-first audit (block 01).
# Stacking, not any single lever, is what turns a 5x overrun back into a line you control. (Ratios illustrative, per block.)
baseline = 1.50 + 0.12 + 0.036            # the measured monthly bill from block 01: rag_answer + summarize + classify ($)
# 1) Prompt-cache the rag_answer prefix (block 02): its big repeated prefix drops to a warm read -> ~70% off this slice.
rag_cached = 1.50 * 0.30
# 2) Gate-route the easy half of rag_answer to a cheap model (block 04): ~40% off what remains, and it clears the eval bar.
rag_routed = rag_cached * 0.60
# 3) Batch the offline summarize slice (block 06): exactly half price on work nobody is waiting on.
summarize_batched = 0.12 * 0.50
classify = 0.036                          # the cheap slice - block 01 said leave it alone, so we do
stacked = rag_routed + summarize_batched + classify
print("Baseline bill (from block 01):    ${:.3f}".format(baseline))
print("  rag_answer $1.500  -> cache -> ${:.3f}  -> gated route -> ${:.3f}".format(rag_cached, rag_routed))
print("  summarize  $0.120  -> batch -> ${:.3f}".format(summarize_batched))
print("  classify   $0.036  -> left alone (cheap slice, not worth optimising)")
print("Stacked bill:                     ${:.3f}   ->  {:.1f}x cheaper than the baseline.".format(stacked, baseline / stacked))
print()
print("No single lever did this: caching + gated routing + batch, each on the slice it fits, COMPOUND to the multiple -")
print("about {:.1f}x, enough to bring a 5x overrun back to roughly the original plan. Measure, apply, gate, and STACK.".format(baseline / stacked))

# Output:
# Baseline bill (from block 01):    $1.656
#   rag_answer $1.500  -> cache -> $0.450  -> gated route -> $0.270
#   summarize  $0.120  -> batch -> $0.060
#   classify   $0.036  -> left alone (cheap slice, not worth optimising)
# Stacked bill:                     $0.366   ->  4.5x cheaper than the baseline.
#
# No single lever did this: caching + gated routing + batch, each on the slice it fits, COMPOUND to the multiple -
# about 4.5x, enough to bring a 5x overrun back to roughly the original plan. Measure, apply, gate, and STACK.

- The baseline is the measured bill from Step 1 — the same Pareto you started with, not a new number.
- Each lever hits the slice it fits: caching the expensive prefix, gated routing on its easy half, batch on the offline work.
- The cheap classify slice is **left alone** — Step 1 said it was not worth optimising, so the stack does not waste effort there.
- Stacked, the levers **compound to about four and a half times cheaper** — enough to bring a five-times overrun back to roughly plan. No single lever did that.

> 💡 **Info**
>
> 🔔 Close the loop: watch the spend, do not just cut itEverything above is *reactive* — the bill surprised you and you cut it. The other half of the ops loop is *proactive*: turn the Step-1 per-feature aggregation into a **scheduled daily job** over your traces (14.1), and put a **budget threshold** on each feature (or an anomaly alert a few standard deviations over its baseline spend). Then a cost spike pages you on day three, while it is still a small number, instead of arriving as a five-times invoice at the end of the month — the same alerting discipline you met for *quality* in Lessons 12.8 and 12.9, now pointed at dollars. Measuring first is not a one-time audit; it is a standing guardrail, and it is what turns a runaway bill from a recurring surprise into a line you actually control.

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now take a real bill apart and put a ceiling on it. What this lesson handled at the level of one service — caching, routing, a GPU or two — becomes a platform problem at scale: multi-region, autoscaled inference, tenant-aware quotas, and cost governance across many teams. That is **platform-scale serving**, and it comes in Lesson 17.4. Before that, the Module 14 capstone in Lesson 14.6 ties tracing, evals, and cost together into one operated system. The primary references are the [Anthropic prompt-caching docs](https://platform.claude.com/docs/en/build-with-claude/prompt-caching), the [OpenAI batch guide](https://platform.openai.com/docs/guides/batch), the [vLLM](https://github.com/vllm-project/vllm) serving project, and [llama.cpp](https://github.com/ggml-org/llama.cpp) for open-model quantization.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Serving& Cost Optimization**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-14_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-14.5.html` - regenerate this notebook whenever the source HTML is updated.*
